# Task 4 – Visualisation & Simple Report

This notebook reloads the models, recomputes metrics if needed, and generates a simple visual comparison of model performance.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline


In [ ]:
# Load cleaned data
df = pd.read_csv("cleaned_spending_data.csv")
FEATURES = ["Gender", "Age", "AnnualIncome"]
TARGET_COL = "SpendingScore"

X = df[FEATURES].copy()
y = df[TARGET_COL].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

categorical_features = ["Gender"]
numeric_features = ["Age", "AnnualIncome"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features),
    ]
)


In [ ]:
# Define models again for comparison
linreg_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LinearRegression())
])

rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    r2 = r2_score(y_test, y_pred)
    return {"name": name, "mae": mae, "rmse": rmse, "r2": r2}

lin_metrics = evaluate_model("Linear Regression", linreg_model, X_train, y_train, X_test, y_test)
rf_metrics = evaluate_model("Random Forest Regressor", rf_model, X_train, y_train, X_test, y_test)

lin_metrics, rf_metrics


In [ ]:
# Build a simple comparison DataFrame
import pandas as pd

results_df = pd.DataFrame([
    lin_metrics,
    rf_metrics
])

results_df


In [ ]:
# Plot R^2 comparison and save image

plt.figure()
plt.bar(results_df["name"], results_df["r2"])
plt.title("Model R^2 Comparison")
plt.xlabel("Model")
plt.ylabel("R^2 Score")

plots_dir = os.path.join("static", "plots")
os.makedirs(plots_dir, exist_ok=True)
plot_path = os.path.join(plots_dir, "model_performance.png")
plt.savefig(plot_path, bbox_inches="tight")
print("Saved performance plot to:", plot_path)
